# Path train data generation

This notebook generates the paths that will be used for the path classification model training. It uses the segmentation predictions generated in the [previous notebook (3)](./03_generate_segmentation_preds.ipynb) to create the paths.

Different path generation methods can be used, but the one used in this notebook is the one detailed in the original paper, which consists in using the segmentation prediction and the ground truth to find the parts of vessels that have been forgotten by the segmentation model. The paths are then generated by connecting the endpoints of these parts to the rest of the vessel tree, and are constituted of a list of coordinates on the image, that follow the an euclidean shortest path between the endpoints.

In [ ]:
import os
from image_segmentation.data import ImageDataset
from utils.available_datasets import available_datasets

dataset_choice = available_datasets["FIVES"]

train_split = dataset_choice.preferred_train_split
data_dir = dataset_choice.data_dir

dataset = ImageDataset(data_dir=data_dir)
stats = dataset.get_dataset_stats(split_name=train_split)
dataset_res = stats["estimated_resolution_mm_per_pixel"]

distances_hparams = dataset.get_dataset_distance_hparams()

In [ ]:
gt_folder = os.path.join(data_dir, "gt")

main_centerlines_folder = os.path.join(data_dir, "centerlines")
os.makedirs(main_centerlines_folder, exist_ok=True)

As stated in the paper, we only consider centerlines with an euclidean length less thant 100 pixels, as longer centerlines are more likely to drift away from the euclidean path bewteen endpoints. Becasue of that, the feature sampling is more likely to be irrelevant and thus those paths are more likely to be wrongly labeled by the model.

If you want to use all the centerlines, just set max_dist to None.

In [ ]:
from math import ceil

max_dist = int(ceil(distances_hparams["max_dist"]))
max_dist_mm = ImageDataset.length_in_pixels_to_mm(max_dist, dataset_res)
print(f"Generating centerlines with max distance of {max_dist} pixels ({max_dist_mm:.2f} mm)")

if max_dist is None:
    centerlines_folder = os.path.join(main_centerlines_folder, f"euclidean_all_centerlines")
else:
    centerlines_folder = os.path.join(main_centerlines_folder, f"euclidean_lt_{max_dist}_centerlines")
os.makedirs(centerlines_folder, exist_ok=True)

In [ ]:
oversampling_max_dist = distances_hparams["oversampling_size"]
oversampling_max_dist_mm = ImageDataset.length_in_pixels_to_mm(oversampling_max_dist, dataset_res)
print(f"Oversampling nodes with max distance of {oversampling_max_dist} pixels ({oversampling_max_dist_mm:.2f} mm)")
n_closest = 5

# Construct the path data (positives and negatives samples)

In [ ]:
from tqdm import tqdm

from path_neural_networks.data.compute_samples import process_case

for i in tqdm(range(len(os.listdir(gt_folder)))):
    process_case(i, data_dir, centerlines_folder, max_dist, oversampling_max_dist, n_closest, display=(i <= 2))

In [ ]:
print(f"Train data centerlines saved to {centerlines_folder}, total {len(os.listdir(centerlines_folder))} centerlines files.")

Now that we have the centerlines data to train the model on, you can continue on the [training path classification model notebook (5)](./05_train_path_classification_model.ipynb)